In [ ]:
import experiment_helper
import igl
from periodic_simulation_setup import *
import json

allowBending = False

name = 'stiffness_resolution'
time_stamp = time.strftime("%Y_%m_%d_%H_%M")
result_folder = 'output/{}/{}'.format(name, time_stamp)
if not os.path.exists(result_folder):
    os.makedirs(result_folder)  

pressure = 0.8
pressure = 2

disableFusedRegionTFT = False

resolutions = np.linspace(0.03, 0.2, 18)
   

In [ ]:
partial_stiffness = []
derivative_over_kappa = []
energy_elastic = []
energy_pressure = []
for res in resolutions:
    h = 2
    w = 0.5
    avg_len = res

    shift = np.array([1.2, 0.])

    ipu, points, segment_edges, m, marker = periodic_unit_helper.get_shifted_dashline(h, w, avg_len, shift, angle = 37 / 180 * np.pi, two_dash = True, opposite_angle = True)

    finalMarkers = np.where(np.array(marker) == 1)[0]
    m, finalMarkers = periodic_unit_helper.shift_and_merge_2D_periodic_mesh(m, finalMarkers)
    m, finalMarkers = periodic_unit_helper.shift_and_merge_2D_periodic_mesh(m, finalMarkers, axis = 1)

    fusedVtx = get_fusedVtx_using_markers(len(m.vertices()), finalMarkers)
    ipu = inflation.InflatablePeriodicUnit(m, fusedVtx = fusedVtx, epsilon = 1e-9)

    viewer = TriMeshViewer(ipu, width=768, height=640)
    viewer.showWireframe(True)

    # Choose strategy for constraining rigid motion
    fixedVars, hessianShift = periodic_unit_helper.get_center_fixedVars(ipu), 0
    if not allowBending:
        fixedVars, hessianShift = [ipu.numVars() - 2], 1e-6
    else:
        fixedVars, hessianShift = [], 1e-6

    ipu.sheet.setUseTensionFieldEnergy(True)
    ipu.sheet.setUseHessianProjectedEnergy(False)
    if (disableFusedRegionTFT):
        ipu.sheet.disableFusedRegionTensionFieldTheory(False)


    ipu.sheet.pressure = pressure

    benchmark.reset()
    print(allowBending, pressure, hessianShift, fixedVars)

    opts.niter = 500
    cr = inflation.inflation_newton(ipu, fixedVars, opts, callback=None, hessianShift = hessianShift)
    viewer.update(scalarField=utils.getStrains(ipu.sheet)[:, 0])
    benchmark.report()

    az_ipu = get_az_ipu_from_ipu(ipu, m, fusedVtx, disableFusedRegionTFT)
    if not allowBending:
        fixedVars, hessianShift = [az_ipu.numVars() - 2], 1e-6
    else:
        fixedVars, hessianShift = [], 1e-6
    opts.niter = 1000
    az_optimizer = inflation.get_inflation_optimizer(az_ipu, fixedVars, opts, callback=None, hessianShift = hessianShift)
    cr = az_optimizer.optimize()
    H = periodic_unit_helper.getNumpyArrayFromCSC(az_ipu.hessian(), reflect = True)
    partial_stiffness.append(H[-2, -2])
    
    bending_stiffness_sample_alpha = inflation.getBendingStiffness(az_ipu, [np.pi / 2], az_optimizer, 1e-10, [])
    partial_stiffness.append(bending_stiffness_sample_alpha[0])
    derivative_over_kappa.append(az_ipu.gradient()[-2])
    energy_elastic.append(az_ipu.energy(energyType = Elastic))    
    energy_pressure.append(az_ipu.energy(energyType = Pressure))

In [ ]:
viewer.show()

In [ ]:
stiffness_values = []
with open("resolution.txt", 'r') as f:
    content = f.readlines()[-18:]
    for line in content:
        numbers = line.strip().split(' ')
        stiffness_values.append([float(numbers[1]), float(numbers[4])])
stiffness_values = np.array(stiffness_values)

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, sharex=True)



ax1.plot(stiffness_values[:, 0])
line, = ax2.plot(stiffness_values[:, 1])
line.set_color("tab:orange")

# zoom-in / limit the view to different portions of the data
ax1.set_ylim(np.min(stiffness_values[:, 0]) - 0.5, np.max(stiffness_values[:, 0] + 0.5))  # outliers only
ax2.set_ylim(np.min(stiffness_values[:, 1]) - 0.5, np.max(stiffness_values[:, 1] + 0.5))  # outliers only

# hide the spines between ax and ax2
ax1.spines.bottom.set_visible(False)
ax2.spines.top.set_visible(False)
ax1.xaxis.tick_top()
ax1.tick_params(labeltop=False)  # don't put tick labels at the top
ax2.xaxis.tick_bottom()

# Now, let's turn towards the cut-out slanted lines.
# We create line objects in axes coordinates, in which (0,0), (0,1),
# (1,0), and (1,1) are the four corners of the axes.
# The slanted lines themselves are markers at those locations, such that the
# lines keep their angle and position, independent of the axes size or scale
# Finally, we need to disable clipping.

d = .5  # proportion of vertical to horizontal extent of the slanted line
kwargs = dict(marker=[(-1, -d), (1, d)], markersize=12,
              linestyle="none", color='k', mec='k', mew=1, clip_on=False)
ax1.plot([0, 1], [0, 0], transform=ax1.transAxes, **kwargs)
ax2.plot([0, 1], [1, 1], transform=ax2.transAxes, **kwargs)




fig.set_size_inches(12, 4)

ax1.title.set_text("Stiffness formula components")

fig.tight_layout()

plt.savefig('stiffness_formula_components_{}.png'.format(name), dpi = 300)

plt.show()



In [ ]:
az_ipu.ipu.get_alpha()

In [ ]:
energy_elastic

In [ ]:
plt.plot(derivative_over_kappa)

In [ ]:
plt.plot(energy_elastic)

In [ ]:
plt.plot(energy_pressure)

In [ ]:
fig, ax = plt.subplots()


plt.plot(np.array(energy_elastic)+ np.array(energy_pressure))




fig.set_size_inches(12, 4)

ax.title.set_text("Potential energy over resolution")

fig.tight_layout()

plt.savefig('energy_over_resolution_{}.png'.format(name), dpi = 300)

plt.show()



In [ ]:
az_ipu.energy(energyType = Elastic)

In [ ]:
az_ipu.energy(energyType = Pressure)

In [ ]:
az_ipu.energy()

In [ ]:
H = periodic_unit_helper.getNumpyArrayFromCSC(az_ipu.hessian(), reflect = True)
# H_F_u = scipy.sparse.csr_matrix(H[:-2, :-2])
H[-2, -2]

In [ ]:
H = periodic_unit_helper.getNumpyArrayFromCSC(az_ipu.hessian(), reflect = True)
# H_F_u = scipy.sparse.csr_matrix(H[:-2, :-2])
H[-2, -2]

In [ ]:
H = periodic_unit_helper.getNumpyArrayFromCSC(az_ipu.hessian(), reflect = True)
# H_F_u = scipy.sparse.csr_matrix(H[:-2, :-2])
H[-2, -2]